# Gemma 4 Developer Agent — first slice (baseline)

Turn Gemma 4 E4B into an offline coding agent that fixes real Python bugs.
The agent loop: read the buggy file, plan a fix, edit it, run the tests,
retry up to 3 times, record pass/fail per bug. No API calls, the model runs
locally on the Kaggle GPU.

**Model:** `google/gemma-4-e4b-it` — the small-model bonus lane ($10,000 for
E2B/E4B; small-model entries stay eligible for the main leaderboard and the
paper track). BF16 weights are ~17.9 GB, so it loads across Kaggle's 2x T4
with `device_map="auto"`. If it OOMs, switch to the 4-bit QAT variant from
the official `google/gemma-4-qat-q4-0` collection on Hugging Face.

**Before running:**
1. Notebook Settings → Accelerator → **GPU T4 x2**.
2. Accept the Gemma license: on Kaggle go to the `google/gemma-4` model page
   and accept, or add your Hugging Face token as a Kaggle secret named
   `HF_TOKEN` (the notebook falls back to it if kagglehub fails).
3. Run all cells top to bottom. The run takes a few minutes after the model
   loads (~2-3 min download on first run).

**What the baseline measures:** pass/fail per bug and attempts used, over
5 small real bugs (off-by-one, wrong comparison, mutable default arg,
missing dict key, wrong sort). Each bug has tests that fail before the fix
and pass after. This is the starting point — later slices add more bugs,
LoRA fine-tuning, and the paper-track write-up.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Enable GPU in Notebook Settings (T4 x2)"


In [ ]:
# Kaggle images usually have these, but install to be safe.
%pip install -q -U transformers accelerate kagglehub


In [ ]:
import sys
from pathlib import Path

# Files embedded by build_notebook.py from the repo (agent.py + bugs/).
# Written to /kaggle/working so the notebook runs from a single upload.
FILES = {
 "agent.py": "\"\"\"Agent loop for the Gemma 4 Developer Agent entry.\n\nThe loop is model-agnostic: the caller passes a ``generate`` callable that\ntakes a prompt string and returns model text. The Kaggle notebook wires\nthat callable up to a locally loaded Gemma 4 E4B model. Nothing here makes\nnetwork calls.\n\nFlow per bug:\n    1. copy the buggy file + its tests into a fresh working directory\n    2. run the tests (they should fail at first)\n    3. ask the model for a fixed version of the file\n    4. write the fix, run the tests again\n    5. retry up to ``max_attempts`` times, then record pass/fail\n\"\"\"\n\nfrom __future__ import annotations\n\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n@dataclass\nclass BugResult:\n    \"\"\"One bug, one row of the baseline table.\"\"\"\n\n    name: str\n    attempts: int = 0\n    passed: bool = False\n    log: list[str] = field(default_factory=list)\n\n\nCODE_BLOCK_RE = re.compile(r\"```(?:python)?\\n(.*?)```\", re.DOTALL)\n\n\ndef extract_code(text: str) -> str:\n    \"\"\"Pull the first python code block out of model output.\n\n    Falls back to the whole output if there is no fenced block, so a model\n    that just prints code still works.\n    \"\"\"\n    match = CODE_BLOCK_RE.search(text)\n    if match:\n        return match.group(1).strip() + \"\\n\"\n    return text.strip() + \"\\n\"\n\n\ndef build_prompt(code: str, test_code: str, test_output: str) -> str:\n    \"\"\"Prompt the model with the buggy file, its tests, and the failure.\"\"\"\n    return (\n        \"You are fixing a small Python bug. Reply with ONLY the complete \"\n        \"corrected file inside one ```python code block, nothing else.\\n\\n\"\n        \"The file is supposed to do what its docstring says. \"\n        \"The tests below describe the correct behaviour.\\n\\n\"\n        \"BUGGY FILE:\\n```python\\n\" + code + \"```\\n\\n\"\n        \"TESTS:\\n```python\\n\" + test_code + \"```\\n\\n\"\n        \"FAILING TEST OUTPUT (last 40 lines):\\n```\\n\"\n        + \"\\n\".join(test_output.strip().splitlines()[-40:])\n        + \"\\n```\\n\\n\"\n        \"CORRECTED FILE:\\n```python\\n\"\n    )\n\n\ndef run_tests(test_path: Path, cwd: Path, timeout: int = 120) -> tuple[bool, str]:\n    \"\"\"Run one test file with pytest. Returns (passed, combined output).\"\"\"\n    proc = subprocess.run(\n        [sys.executable, \"-m\", \"pytest\", \"-q\", \"-p\", \"no:cacheprovider\", str(test_path)],\n        cwd=cwd,\n        capture_output=True,\n        text=True,\n        timeout=timeout,\n    )\n    return proc.returncode == 0, proc.stdout + proc.stderr\n\n\ndef run_agent(\n    generate,\n    bug_file: Path,\n    test_file: Path,\n    work_root: Path,\n    max_attempts: int = 3,\n) -> BugResult:\n    \"\"\"Run the fix loop for one bug. Edits only a copy, never the original.\"\"\"\n    name = bug_file.stem\n    result = BugResult(name=name)\n    workdir = work_root / name\n    testdir = workdir / \"tests\"\n    testdir.mkdir(parents=True, exist_ok=True)\n\n    work_bug = workdir / bug_file.name\n    work_test = testdir / test_file.name\n    shutil.copy(bug_file, work_bug)\n    shutil.copy(test_file, work_test)\n    # conftest.py makes the bug module importable from the tests dir.\n    conftest = test_file.parent / \"conftest.py\"\n    if conftest.exists():\n        shutil.copy(conftest, testdir / \"conftest.py\")\n\n    code = work_bug.read_text()\n    test_code = work_test.read_text()\n\n    passed, output = run_tests(work_test, cwd=workdir)\n    result.log.append(\"attempt 0 (original): \" + (\"PASS\" if passed else \"FAIL\"))\n    if passed:\n        result.passed = True\n        return result\n\n    for attempt in range(1, max_attempts + 1):\n        result.attempts = attempt\n        prompt = build_prompt(code, test_code, output)\n        try:\n            fixed = extract_code(generate(prompt))\n        except Exception as exc:  # model call failed, count as an attempt\n            result.log.append(f\"attempt {attempt}: generate() raised {exc!r}\")\n            continue\n        work_bug.write_text(fixed)\n        code = fixed\n        passed, output = run_tests(work_test, cwd=workdir)\n        result.log.append(f\"attempt {attempt}: \" + (\"PASS\" if passed else \"FAIL\"))\n        if passed:\n            result.passed = True\n            return result\n\n    return result\n\n\ndef run_all(generate, bugs_dir: Path, work_root: Path, max_attempts: int = 3) -> list[BugResult]:\n    \"\"\"Run the agent over every bug_*.py in bugs_dir (sorted by name).\"\"\"\n    results = []\n    for bug_file in sorted(bugs_dir.glob(\"bug_*.py\")):\n        test_file = bugs_dir / \"tests\" / f\"test_{bug_file.stem}.py\"\n        if not test_file.exists():\n            print(f\"SKIP {bug_file.name}: no {test_file.name}\")\n            continue\n        print(f\"--- {bug_file.name} ---\")\n        res = run_agent(generate, bug_file, test_file, work_root, max_attempts)\n        print(f\"    {'PASS' if res.passed else 'FAIL'} after {res.attempts} attempt(s)\")\n        results.append(res)\n    return results\n\n\ndef results_table(results: list[BugResult]) -> str:\n    \"\"\"Markdown table for baseline_results.md.\"\"\"\n    lines = [\n        \"| Bug | Attempts | Result |\",\n        \"| --- | --- | --- |\",\n    ]\n    for res in results:\n        lines.append(f\"| {res.name} | {res.attempts} | {'pass' if res.passed else 'fail'} |\")\n    passed = sum(r.passed for r in results)\n    lines.append(\"\")\n    lines.append(f\"Pass rate: {passed}/{len(results)}\")\n    return \"\\n\".join(lines) + \"\\n\"\n",
 "bugs/bug_01_off_by_one.py": "\"\"\"Add up a list of numbers.\n\nBug: the total is wrong, it misses the last number.\n\"\"\"\n\n\ndef total(xs):\n    s = 0\n    for i in range(len(xs) - 1):\n        s += xs[i]\n    return s\n",
 "bugs/bug_02_wrong_comparison.py": "\"\"\"Check whether someone is an adult (18 or older).\"\"\"\n\n\ndef is_adult(age):\n    return age > 18\n",
 "bugs/bug_03_mutable_default.py": "\"\"\"Add an item to a shopping cart and return the cart.\n\nEach call without a cart should start from an empty cart.\n\"\"\"\n\n\ndef add_item(item, cart=[]):\n    cart.append(item)\n    return cart\n",
 "bugs/bug_04_missing_key.py": "\"\"\"Look up the price of an item.\n\nUnknown items cost 0 instead of crashing.\n\"\"\"\n\nPRICES = {\"apple\": 10, \"banana\": 5, \"orange\": 8}\n\n\ndef get_price(item):\n    return PRICES[item]\n",
 "bugs/bug_05_wrong_sort.py": "\"\"\"Return the 3 highest scores, highest first.\"\"\"\n\n\ndef top_scores(scores):\n    return sorted(scores)[:3]\n",
 "bugs/tests/conftest.py": "\"\"\"Makes the parent bugs/ directory importable so tests can import the bug modules.\"\"\"\nimport os\nimport sys\n\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), \"..\")))\n",
 "bugs/tests/test_bug_01_off_by_one.py": "from bug_01_off_by_one import total\n\n\ndef test_total_all_numbers():\n    assert total([1, 2, 3]) == 6\n\n\ndef test_total_single():\n    assert total([7]) == 7\n\n\ndef test_total_empty():\n    assert total([]) == 0\n",
 "bugs/tests/test_bug_02_wrong_comparison.py": "from bug_02_wrong_comparison import is_adult\n\n\ndef test_eighteen_is_adult():\n    assert is_adult(18) is True\n\n\ndef test_twenty_is_adult():\n    assert is_adult(20) is True\n\n\ndef test_seventeen_is_not_adult():\n    assert is_adult(17) is False\n",
 "bugs/tests/test_bug_03_mutable_default.py": "from bug_03_mutable_default import add_item\n\n\ndef test_first_call():\n    assert add_item(\"apple\") == [\"apple\"]\n\n\ndef test_carts_are_independent():\n    a = add_item(\"apple\")\n    b = add_item(\"banana\")\n    assert a == [\"apple\"]\n    assert b == [\"banana\"]\n\n\ndef test_passed_cart_is_used():\n    assert add_item(\"pear\", [\"plum\"]) == [\"plum\", \"pear\"]\n",
 "bugs/tests/test_bug_04_missing_key.py": "from bug_04_missing_key import get_price\n\n\ndef test_known_item():\n    assert get_price(\"apple\") == 10\n\n\ndef test_unknown_item_is_zero():\n    assert get_price(\"mango\") == 0\n\n\ndef test_another_unknown_item():\n    assert get_price(\"kiwi\") == 0\n",
 "bugs/tests/test_bug_05_wrong_sort.py": "from bug_05_wrong_sort import top_scores\n\n\ndef test_top_three_highest_first():\n    assert top_scores([5, 1, 9, 3]) == [9, 5, 3]\n\n\ndef test_fewer_than_three():\n    assert top_scores([2, 7]) == [7, 2]\n\n\ndef test_already_sorted():\n    assert top_scores([10, 8, 6, 4]) == [10, 8, 6]\n"
}

for rel, content in FILES.items():
    path = Path("/kaggle/working") / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)

sys.path.insert(0, "/kaggle/working")
import agent

# Smoke test: the harness works and the first bug really fails before any fix.
passed, out = agent.run_tests(
    Path("/kaggle/working/bugs/tests/test_bug_01_off_by_one.py"),
    cwd=Path("/kaggle/working/bugs"),
)
print("smoke test, bug_01 original:", "PASS (unexpected!)" if passed else "FAIL (expected)")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-4-e4b-it"  # instruction-tuned E4B

model_path = None
try:
    import kagglehub
    model_path = kagglehub.model_download("google/gemma-4/transformers/e4b-it")
    print("downloaded via kagglehub:", model_path)
except Exception as e:
    print("kagglehub failed, falling back to Hugging Face:", e)

load_from = model_path or MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(load_from)
model = AutoModelForCausalLM.from_pretrained(
    load_from,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("model loaded:", load_from)


In [ ]:
def generate(prompt: str) -> str:
    """Gemma-backed generate() for agent.run_all. Low temperature: fixes
    should be mostly deterministic."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.2,
            top_p=0.95,
            do_sample=True,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )


print(generate("Write one line of Python that adds 1 and 2. Reply with code only."))


In [ ]:
from pathlib import Path

results = agent.run_all(
    generate,
    Path("/kaggle/working/bugs"),
    Path("/kaggle/working/runs"),
    max_attempts=3,
)
print()
print(agent.results_table(results))


In [ ]:
from pathlib import Path

md = (
    "# Baseline results\n\n"
    "Model: google/gemma-4-e4b-it (bf16, Kaggle 2x T4)\n"
    "Date: (fill in)\n\n"
    + agent.results_table(results)
)
Path("/kaggle/working/baseline_results.md").write_text(md)
print(md)
print("saved to /kaggle/working/baseline_results.md — copy it back to the repo")


## Next steps

- **More bugs:** add harder ones (wrong loop bounds in real code, type errors,
  broken imports) to `bugs/` and re-run. The loop needs no changes.
- **LoRA fine-tuning:** fine-tune E4B on before/after fix pairs (the
  competition allows LoRA; the small-model bonus still applies). Start from
  the fixes this agent produces, corrected by hand.
- **Paper track:** the paper topics include new tasks and eval datasets —
  this 5-bug set is the seed of one. Log every attempt, not just passes.
- **Submission:** the final Kaggle submission is the notebook itself plus a
  short write-up. Deadline Thu 3 Dec 2026, 05:29 IST.